In [14]:
# ====================================================
# NeurIPS 2025 - Open Polymer Prediction
# Phase 2: Maximally Upgraded Baseline
# ====================================================

import pandas as pd
import numpy as np
from tqdm import tqdm

from rdkit import Chem
from mordred import Calculator, descriptors
from rdkit.Chem import AllChem

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error


In [15]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


Train shape: (7973, 7)
Test shape: (3, 2)


,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,NaN,0.370410,NaN,NaN,NaN
2,388772,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,NaN,0.378860,NaN,NaN,NaN
3,519416,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,NaN,0.387324,NaN,NaN,NaN
4,539187,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,NaN,0.355470,NaN,NaN,NaN


In [23]:
# -----------------------
# 2) Mordred descriptors (batch mode, robust)
# -----------------------
from rdkit import Chem
from mordred import Calculator, descriptors
import pandas as pd
import numpy as np

def smiles_to_mol(s):
    if not isinstance(s, str):
        return None
    s2 = s.replace("*","C")  # simple handling for polymer wildcard
    try:
        return Chem.MolFromSmiles(s2)
    except:
        return None

calc = Calculator(descriptors, ignore_3D=True)

# Build RDKit mol lists (keep alignment with train/test)
train_mols = [smiles_to_mol(s) for s in train["SMILES"]]
test_mols  = [smiles_to_mol(s) for s in test["SMILES"]]

# Batch-compute Mordred
train_desc = calc.pandas(train_mols)   # returns mixed dtypes (numbers + Missing)
test_desc  = calc.pandas(test_mols)

# Coerce to numeric (Missing -> NaN), drop all-NaN & constant cols (by TRAIN)
train_desc = train_desc.apply(pd.to_numeric, errors="coerce")
test_desc  = test_desc.apply(pd.to_numeric, errors="coerce")

train_desc = train_desc.dropna(axis=1, how="all")
nunique = train_desc.nunique(dropna=True)
train_desc = train_desc.loc[:, nunique > 1]

# Align TEST to TRAIN columns
test_desc = test_desc.reindex(columns=train_desc.columns)

# Median impute using TRAIN medians
med = train_desc.median(numeric_only=True)
train_desc_clean = train_desc.fillna(med)
test_desc_clean  = test_desc.fillna(med)

print("Mordred features ->")
print("  train_desc_clean:", train_desc_clean.shape)
print("  test_desc_clean :", test_desc_clean.shape)


  0%|          | 4/7973 [00:02<1:56:05,  1.14it/s]

/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  0%|          | 25/7973 [00:02<31:46,  4.17it/s] 

/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  1%|          | 43/7973 [00:03<07:23, 17.89it/s]

...

/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  6%|▌         | 447/7973 [00:20<04:41, 26.71it/s]

/home/rohit/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 3/3 [00:00<00:00, 10.34it/s]


Mordred features ->
  train_desc_clean: (7973, 1412)
  test_desc_clean : (3, 1412)


In [25]:
# -----------------------
# 3) Optional extra features (count Morgan FP + basic RDKit 2D) and concatenate
# -----------------------
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

def morgan_count_fp(mol, radius=2, nBits=2048):
    """Count-based Morgan fingerprint as dense array"""
    if mol is None:
        return np.zeros(nBits, dtype=float)
    fp = AllChem.GetHashedMorganFingerprint(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=int)
    for idx, count in fp.GetNonzeroElements().items():
        arr[idx % nBits] = count
    return arr.astype(float)

def rdkit_basic(mol):
    """Basic RDKit 2D descriptors with safeguards"""
    if mol is None:
        return np.array([np.nan]*8, dtype=float)
    try:
        return np.array([
            Descriptors.MolWt(mol),
            Descriptors.TPSA(mol),
            Descriptors.MolLogP(mol),
            rdMolDescriptors.CalcNumRings(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcNumAliphaticRings(mol),
            rdMolDescriptors.CalcNumHBD(mol),  # donors
            rdMolDescriptors.CalcNumHBA(mol),  # acceptors
        ], dtype=float)
    except Exception:
        return np.array([np.nan]*8, dtype=float)

def build_extra_features(mols):
    X_fp = np.vstack([morgan_count_fp(m) for m in mols])
    X_rd = np.vstack([rdkit_basic(m) for m in mols])
    # Impute NaNs in RDKit descriptors
    col_med = np.nanmedian(X_rd, axis=0)
    inds = np.where(np.isnan(X_rd))
    X_rd[inds] = np.take(col_med, inds[1])
    return np.hstack([X_fp, X_rd])

# Toggle for using extra features
USE_EXTRA = True  

if USE_EXTRA:
    X_train_extra = build_extra_features(train_mols)
    X_test_extra  = build_extra_features(test_mols)

    X_train_all = np.hstack([train_desc_clean.values, X_train_extra])
    X_test_all  = np.hstack([test_desc_clean.values,  X_test_extra])

    print("Combined feature shapes ->", X_train_all.shape, X_test_all.shape)
else:
    X_train_all = train_desc_clean.values
    X_test_all  = test_desc_clean.values
    print("Using Mordred only ->", X_train_all.shape, X_test_all.shape)


[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerator
[06:36:21] DEPRECATION WARNING: please use MorganGenerat

Combined feature shapes -> (7973, 3468) (3, 3468)


In [30]:
# -----------------------
# 4) Competition Metric (wMAE) - robust
# -----------------------
from sklearn.metrics import mean_absolute_error

TARGETS = ["Tg","FFV","Tc","Density","Rg"]

def competition_wmae(y_true_dict, y_pred_dict):
    maes, weights, total_weight = {}, {}, 0.0
    for target in TARGETS:
        y_true_full = np.asarray(y_true_dict[target])
        y_pred_full = np.asarray(y_pred_dict[target])

        # Valid where both y_true exists and y_pred is finite
        mask = (~np.isnan(y_true_full)) & np.isfinite(y_pred_full)
        if mask.sum() == 0:
            continue

        y_true = y_true_full[mask]
        y_pred = y_pred_full[mask]

        mae = mean_absolute_error(y_true, y_pred)
        value_range = np.nanmax(y_true) - np.nanmin(y_true)
        if value_range == 0:
            value_range = 1.0

        weight = (1.0 / np.sqrt(mask.sum())) / value_range
        maes[target] = mae
        weights[target] = weight
        total_weight += weight

    # Normalize weights
    for t in list(weights.keys()):
        weights[t] = weights[t] / total_weight if total_weight > 0 else 0.0

    wmae = sum(maes[t] * weights[t] for t in maes)
    return wmae, maes


In [31]:
# -----------------------
# 5) Training with CV (LightGBM; correct indexing)
# -----------------------
import lightgbm as lgb
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Prepare holders
oof_preds = {t: np.full(len(train), np.nan, dtype=float) for t in TARGETS}
test_preds = {t: np.zeros(len(test), dtype=float) for t in TARGETS}

for target in TARGETS:
    print(f"\nTraining for target: {target}")

    # Keep original indices (no reset_index!)
    labeled_idx = train.index[~train[target].isna()].to_numpy()
    y = train.loc[labeled_idx, target].to_numpy()

    # Subset features using original indices
    X = X_train_all[labeled_idx]
    X_te = X_test_all

    fold_oof = np.zeros(len(labeled_idx), dtype=float)
    fold_te  = np.zeros((len(test), kf.get_n_splits()), dtype=float)

    for i, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        model = lgb.LGBMRegressor(
            n_estimators=5000,
            learning_rate=0.01,
            num_leaves=128,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=i
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="mae",
            callbacks=[
                lgb.early_stopping(200),
                lgb.log_evaluation(200)
            ]
        )

        fold_oof[val_idx] = model.predict(X_val, num_iteration=model.best_iteration_)
        fold_te[:, i] = model.predict(X_te, num_iteration=model.best_iteration_)

    # Write OOF back to the correct rows in the full-length vector
    oof_preds[target][labeled_idx] = fold_oof
    test_preds[target] = fold_te.mean(axis=1)

# Evaluate OOF wMAE
wmae_score, maes = competition_wmae(
    {t: train[t].values for t in TARGETS},
    oof_preds
)
print("\nValidation Results:")
print("  wMAE:", wmae_score)
print("  Per-target MAEs:", maes)



Training for target: Tg
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012430 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 142427
[LightGBM] [Info] Number of data points in the train set: 408, number of used features: 1356
[LightGBM] [Info] Start training from score 98.747357
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011764 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 142382
[LightGBM] [Info] Number of data points in the train set: 409, number of used features: 1356
[LightGBM] [Info] Start training from score 99.663606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012631 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 142488
[LightGBM] [Info] Number of data points in the train set: 409, number of used features: 1360
[LightGBM] [Info] Start training from score 94.149225
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

...

/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012687 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 162723
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 1322
[LightGBM] [Info] Start training from score 16.501562
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013737 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 163690
[LightGBM] [Info] Number of data points in the train set: 492, number of used features: 1321
[LightGBM] [Info] Start training from score 16.347690
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/rohit/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [32]:
# -----------------------
# 6) Submission
# -----------------------
submission = sample.copy()
for target in TARGETS:
    submission[target] = test_preds[target]

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())


Saved submission.csv


,id,Tg,FFV,Tc,Density,Rg
0,1109053969,130.931607,0.373033,0.174694,1.178823,19.921362
1,1422188626,153.280700,0.377296,0.243550,1.067995,19.472386
2,2032016830,77.150468,0.350990,0.292772,1.118775,21.759989


In [33]:
import pandas as pd, numpy as np

sub = pd.read_csv("submission.csv")
test_ids = pd.read_csv("test.csv")["id"]

# Column/order check
assert list(sub.columns) == ["id","Tg","FFV","Tc","Density","Rg"]

# Id coverage & duplicates
assert set(sub["id"]) == set(test_ids), "submission ids != test ids"
assert sub["id"].is_unique, "duplicate ids in submission"

# Missing / non-numeric checks
assert not sub.isna().any().any(), "NaNs found in submission"
for c in ["Tg","FFV","Tc","Density","Rg"]:
    assert np.issubdtype(sub[c].dtype, np.number), f"{c} must be numeric"

print("submission.csv looks good ✅")
sub.head()


submission.csv looks good ✅


,id,Tg,FFV,Tc,Density,Rg
0,1109053969,130.931607,0.373033,0.174694,1.178823,19.921362
1,1422188626,153.280700,0.377296,0.243550,1.067995,19.472386
2,2032016830,77.150468,0.350990,0.292772,1.118775,21.759989


In [34]:
# --- WHERE TO SAVE ---
ART_DIR = "artifacts"
import os, joblib, numpy as np, pandas as pd
os.makedirs(ART_DIR, exist_ok=True)

# We assume you already built these earlier in this notebook:
# - train_desc_clean, test_desc_clean  (Mordred cleaned)
# - X_train_extra, X_test_extra        (optional RDKit extras)
# - USE_EXTRA (True/False)
# - oof_preds, test_preds, TARGETS     (from training loop)

# Save Mordred feature column names (order matters)
mordred_cols = list(train_desc_clean.columns)
joblib.dump(mordred_cols, f"{ART_DIR}/mordred_cols.pkl")

# Save medians used to fill Mordred NaNs (as a pandas Series to preserve index)
mordred_medians = train_desc_clean.median(numeric_only=True)
joblib.dump(mordred_medians, f"{ART_DIR}/mordred_medians.pkl")

# Save whether we used extra RDKit features and their dimensionality
cfg = {
    "USE_EXTRA": bool('X_train_extra' in globals()),
    "EXTRA_DIM": int(X_train_extra.shape[1]) if 'X_train_extra' in globals() else 0,
    "FP_BITS": int(X_train_extra.shape[1] - 8) if 'X_train_extra' in globals() else 2048,
}
joblib.dump(cfg, f"{ART_DIR}/config.pkl")

print("Saved feature schema & config to:", ART_DIR)


Saved feature schema & config to: artifacts


In [35]:
# Re-run your CV loop? No. We just save the fitted models while training.
# If you didn't keep references, retrain quickly OR modify your training loop to joblib.dump each fold model.

# Example: assuming you retrain quickly to save models (same params)
import lightgbm as lgb
from sklearn.model_selection import KFold

MODEL_DIR = f"{ART_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for target in TARGETS:
    print(f"Saving models for {target}")
    labeled_idx = train.index[~train[target].isna()].to_numpy()
    y = train.loc[labeled_idx, target].to_numpy()
    X = X_train_all[labeled_idx]

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        mdl = lgb.LGBMRegressor(
            n_estimators=5000, learning_rate=0.01,
            num_leaves=128, subsample=0.8, colsample_bytree=0.8,
            random_state=100+fold, objective="mae"
        )
        mdl.fit(
            X[tr_idx], y[tr_idx],
            eval_set=[(X[val_idx], y[val_idx])],
            eval_metric="mae",
            callbacks=[lgb.early_stopping(200)]
        )
        joblib.dump(mdl, f"{MODEL_DIR}/{target}_fold{fold}.pkl")

print("All models saved to:", MODEL_DIR)


Saving models for Tg
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012863 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 142427
[LightGBM] [Info] Number of data points in the train set: 408, number of used features: 1356
[LightGBM] [Info] Start training from score 72.122604
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos